# Dijet generator-smearing effect

Compare nominal generator, generator-level smearing, eta-dependent generator-level smearing, and reconstructed dijet pseudorapidity distributions for one configured jet $\eta_{CM}$ acceptance and every interval in `DIJET_PTAVE_BINS`. Full $\eta_{CM}^{dijet}$ projections are normalized as densities before comparison; forward and backward projections remain unnormalized when their ratio is formed. Distribution overlays, ratios to nominal Gen, and ratios to eta-dependent smeared Gen are drawn on separate canvases.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_io import (
    resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.plotting import draw_overlay

## Configuration

`CURVES` is the user-facing ordered list of distributions. Every key is a template whose `{eta_cut_index}` field selects the stored eta-acceptance histogram. `ETA_CUT_INDEX=5` corresponds to the nominal $|\eta_{CM}^{jet}|<1.9$ acceptance. Projection intervals are half-open. `NORMALIZATION='bin_width'` gives the full overlays the displayed $1/N\,dN/d\eta_{CM}^{dijet}$ density. Independent ROOT error propagation is the default because forward and backward weighted populations and normalized shape ratios are not binomial subset efficiencies.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS = tuple(DIJET_PTAVE_BINS)
REBIN_ETA = 2
NORMALIZATION = 'integral'   # required for 1/N dN/deta density overlays
RATIO_OPTION = 'B'             # '' for independent, 'B' for binomial errors
FULL_RATIO_RANGE = (0.85, 1.3)
FB_RANGE = (0.9, 1.30)
FB_RATIO_RANGE = (0.9, 1.3)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_smearing_effect'

CURVES = (
    DijetClosureCurve(
        'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
        'hGenDijetPtEtaForward_{eta_cut_index}',
        'hGenDijetPtEtaBackward_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Gen (JER x 1.0)', 'hGenDijetDefPtEtaCM_{eta_cut_index}',
        'hGenDijetDefPtEtaForward_{eta_cut_index}',
        'hGenDijetDefPtEtaBackward_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Gen (JER #eta-dep.)',
        'hGenDijetDefExtraPtEtaCM_{eta_cut_index}',
        'hGenDijetDefExtraPtEtaForward_{eta_cut_index}',
        'hGenDijetDefExtraPtEtaBackward_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'Reco', 'hRecoDijetPtEtaCM_{eta_cut_index}',
        'hRecoDijetPtEtaForward_{eta_cut_index}',
        'hRecoDijetPtEtaBackward_{eta_cut_index}',
    ),
)
NOMINAL = 'Gen'
ETA_DEPENDENT_NOMINAL = 'Gen (JER #eta-dep.)'
STYLE_INDICES = {
    'Gen': 2, 'Gen (JER x 1.0)': 3,
    'Gen (JER #eta-dep.)': 1, 'Reco': 0,
}

if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
curve_labels = {curve.label for curve in CURVES}
if NOMINAL not in curve_labels:
    raise ValueError(f'NOMINAL={NOMINAL!r} is not present in CURVES')
if ETA_DEPENDENT_NOMINAL not in curve_labels:
    raise ValueError(
        f'ETA_DEPENDENT_NOMINAL={ETA_DEPENDENT_NOMINAL!r} is not present in CURVES'
    )
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]

## Resolve the input

Combined and direction-specific samples use the common repository file resolvers. Histogram existence, TH2 type, and common projected eta binning are validated while each pTave interval is built.

In [3]:
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

PosixPath('/Users/gnigmat/cernbox/ana/pPb8160/embedding/embedding_jetId.root')

## Full-shape and forward/backward comparisons

For each configured pTave interval, six independent canvases are produced: the normalized full-shape overlay, full-shape ratios to Gen and to eta-dependent smeared Gen, the unnormalized forward/backward-ratio overlay, and the corresponding double ratios to both references. Ratio histograms are never placed in a lower pad.

In [4]:
comparison_results = {}
eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
eta_cut_tag = int(round(10.0 * ETA_CUT))

for ptave_range in PTAVE_BINS:
    low, high = ptave_range
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    common_tag = (
        f'{GENERATOR}_{DIRECTION}_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
    )
    eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
        INPUT_FILE, CURVES, eta_cut_index=ETA_CUT_INDEX,
        ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
        normalization=NORMALIZATION, ratio_option=RATIO_OPTION,
    )
    eta_to_gen = {
        label: ratio_to_nominal(
            histogram, eta_shapes[NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_to_gen',
            option=RATIO_OPTION,
        )
        for label, histogram in eta_shapes.items() if label != NOMINAL
    }
    fb_to_gen = {
        label: ratio_to_nominal(
            histogram, fb_ratios[NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_fb_to_gen',
            option=RATIO_OPTION,
        )
        for label, histogram in fb_ratios.items() if label != NOMINAL
    }
    eta_to_eta_dependent = {
        label: ratio_to_nominal(
            histogram, eta_shapes[ETA_DEPENDENT_NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_to_gen_eta_dep',
            option=RATIO_OPTION,
        )
        for label, histogram in eta_shapes.items()
        if label != ETA_DEPENDENT_NOMINAL
    }
    fb_to_eta_dependent = {
        label: ratio_to_nominal(
            histogram, fb_ratios[ETA_DEPENDENT_NOMINAL],
            name=f'h_{common_tag}_{label.replace(" ", "_")}_fb_to_gen_eta_dep',
            option=RATIO_OPTION,
        )
        for label, histogram in fb_ratios.items()
        if label != ETA_DEPENDENT_NOMINAL
    }
    annotations = (
        GENERATOR.capitalize(),
        'Gen and Reco dijets',
        'CM frame',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} > 50 GeV',
        'p_{T}^{SubLead} > 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    canvases = {
        'eta_overlay': draw_overlay(
            eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
            y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
            annotations=annotations, grid=DRAW_GRID, headroom=1.55,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{common_tag}_full_overlay.pdf',
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay',
        ),
        'eta_ratio': draw_overlay(
            eta_to_gen, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Ratio to Gen', x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{common_tag}_full_ratio_to_gen.pdf',
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_ratio',
        ),
        'eta_ratio_to_eta_dependent': draw_overlay(
            eta_to_eta_dependent, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Ratio to Gen (JER #eta-dep.)', x_range=eta_x_range,
            y_range=FULL_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{common_tag}_full_ratio_to_gen_eta_dep.pdf',
            save_png=SAVE_PNG,
            canvas_name=f'{common_tag}_full_ratio_to_gen_eta_dep',
        ),
        'fb_overlay': draw_overlay(
            fb_ratios, title='', x_title='#eta_{CM}^{dijet}',
            y_title='Forward / Backward', x_range=fb_x_range,
            y_range=FB_RANGE, reference_y=1.0, annotations=annotations,
            grid=DRAW_GRID, style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{common_tag}_fb_overlay.pdf',
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay',
        ),
        'fb_ratio': draw_overlay(
            fb_to_gen, title='', x_title='#eta_{CM}^{dijet}',
            y_title='(Forward / Backward) ratio to Gen',
            x_range=fb_x_range, y_range=FB_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{common_tag}_fb_ratio_to_gen.pdf',
            save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_ratio',
        ),
        'fb_ratio_to_eta_dependent': draw_overlay(
            fb_to_eta_dependent, title='', x_title='#eta_{CM}^{dijet}',
            y_title='(Forward / Backward) ratio to Gen (JER #eta-dep.)',
            x_range=fb_x_range, y_range=FB_RATIO_RANGE, reference_y=1.0,
            annotations=annotations, grid=DRAW_GRID,
            style_indices=STYLE_INDICES,
            output=OUTPUT_DIR / f'{common_tag}_fb_ratio_to_gen_eta_dep.pdf',
            save_png=SAVE_PNG,
            canvas_name=f'{common_tag}_fb_ratio_to_gen_eta_dep',
        ),
    }
    comparison_results[ptave_range] = {
        'eta_shapes': eta_shapes, 'eta_to_gen': eta_to_gen,
        'eta_to_eta_dependent': eta_to_eta_dependent,
        'forward_backward': fb_ratios, 'forward_backward_to_gen': fb_to_gen,
        'forward_backward_to_eta_dependent': fb_to_eta_dependent,
        'canvases': canvases, 'keys': selected_keys,
    }
    print(common_tag, selected_keys)
    for canvas in canvases.values():
        display(canvas)

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_60_80_full_overlay.pdf has been created


embedding_combined_etaCM_19_ptave_60_80 {'Gen': {'cm': 'hGenDijetPtEtaCM_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}, 'Gen (JER x 1.0)': {'cm': 'hGenDijetDefPtEtaCM_5', 'forward': 'hGenDijetDefPtEtaForward_5', 'backward': 'hGenDijetDefPtEtaBackward_5'}, 'Gen (JER #eta-dep.)': {'cm': 'hGenDijetDefExtraPtEtaCM_5', 'forward': 'hGenDijetDefExtraPtEtaForward_5', 'backward': 'hGenDijetDefExtraPtEtaBackward_5'}, 'Reco': {'cm': 'hRecoDijetPtEtaCM_5', 'forward': 'hRecoDijetPtEtaForward_5', 'backward': 'hRecoDijetPtEtaBackward_5'}}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_60_80_full_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_60_80_full_ratio_to_gen_eta_dep.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_60_80_fb_overlay.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_60_80_fb_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_60_80_fb_ratio_to_gen_eta_dep.pdf has been created


embedding_combined_etaCM_19_ptave_120_180 {'Gen': {'cm': 'hGenDijetPtEtaCM_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}, 'Gen (JER x 1.0)': {'cm': 'hGenDijetDefPtEtaCM_5', 'forward': 'hGenDijetDefPtEtaForward_5', 'backward': 'hGenDijetDefPtEtaBackward_5'}, 'Gen (JER #eta-dep.)': {'cm': 'hGenDijetDefExtraPtEtaCM_5', 'forward': 'hGenDijetDefExtraPtEtaForward_5', 'backward': 'hGenDijetDefExtraPtEtaBackward_5'}, 'Reco': {'cm': 'hRecoDijetPtEtaCM_5', 'forward': 'hRecoDijetPtEtaForward_5', 'backward': 'hRecoDijetPtEtaBackward_5'}}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_120_180_full_overlay.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_120_180_full_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_120_180_full_ratio_to_gen_eta_dep.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_120_180_fb_overlay.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_120_180_fb_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf f

embedding_combined_etaCM_19_ptave_200_300 {'Gen': {'cm': 'hGenDijetPtEtaCM_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}, 'Gen (JER x 1.0)': {'cm': 'hGenDijetDefPtEtaCM_5', 'forward': 'hGenDijetDefPtEtaForward_5', 'backward': 'hGenDijetDefPtEtaBackward_5'}, 'Gen (JER #eta-dep.)': {'cm': 'hGenDijetDefExtraPtEtaCM_5', 'forward': 'hGenDijetDefExtraPtEtaForward_5', 'backward': 'hGenDijetDefExtraPtEtaBackward_5'}, 'Reco': {'cm': 'hRecoDijetPtEtaCM_5', 'forward': 'hRecoDijetPtEtaForward_5', 'backward': 'hRecoDijetPtEtaBackward_5'}}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_200_300_full_overlay.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_200_300_full_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_200_300_full_ratio_to_gen_eta_dep.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_200_300_fb_overlay.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_200_300_fb_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf f

embedding_combined_etaCM_19_ptave_300_500 {'Gen': {'cm': 'hGenDijetPtEtaCM_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}, 'Gen (JER x 1.0)': {'cm': 'hGenDijetDefPtEtaCM_5', 'forward': 'hGenDijetDefPtEtaForward_5', 'backward': 'hGenDijetDefPtEtaBackward_5'}, 'Gen (JER #eta-dep.)': {'cm': 'hGenDijetDefExtraPtEtaCM_5', 'forward': 'hGenDijetDefExtraPtEtaForward_5', 'backward': 'hGenDijetDefExtraPtEtaBackward_5'}, 'Reco': {'cm': 'hRecoDijetPtEtaCM_5', 'forward': 'hRecoDijetPtEtaForward_5', 'backward': 'hRecoDijetPtEtaBackward_5'}}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_300_500_full_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_300_500_full_ratio_to_gen_eta_dep.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_300_500_fb_overlay.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_300_500_fb_ratio_to_gen.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/dijet_smearing_effect/embedding_combined_etaCM_19_ptave_300_500_fb_ratio_to_gen_eta_dep.pdf has been created


## Numerical audit

Report the bin-width-normalized full-shape integrals and finite extrema of every comparison to Gen and eta-dependent smeared Gen. This exposes bins outside the configured display ranges without modifying their contents.

In [5]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None

for ptave_range, result in comparison_results.items():
    print(f'\npTave interval {ptave_range}, eta cut {ETA_CUT:g}')
    for label in (curve.label for curve in CURVES):
        print(
            f'  {label:27s} full width integral='
            f'{result["eta_shapes"][label].Integral("width"):.8g}, '
            f'F/B range={finite_nonzero_range(result["forward_backward"][label])}'
        )
        if label != NOMINAL:
            print(
                f'    full/Gen range='
                f'{finite_nonzero_range(result["eta_to_gen"][label])}, '
                f'(F/B)/(F/B)_Gen range='
                f'{finite_nonzero_range(result["forward_backward_to_gen"][label])}'
            )
        if label != ETA_DEPENDENT_NOMINAL:
            print(
                f'    full/Gen-eta-dep range='
                f'{finite_nonzero_range(result["eta_to_eta_dependent"][label])}, '
                f'(F/B)/(F/B)_Gen-eta-dep range='
                f'{finite_nonzero_range(result["forward_backward_to_eta_dependent"][label])}'
            )



pTave interval (60, 80), eta cut 1.9
  Gen                         full width integral=0.2, F/B range=(0.7980325838455499, 1.0356797654707048)
    full/Gen-eta-dep range=(0.9691601228796123, 1.048372024912559), (F/B)/(F/B)_Gen-eta-dep range=(0.9244429456808962, 1.0059902275928805)
  Gen (JER x 1.0)             full width integral=0.2, F/B range=(0.8831736757539927, 1.030908554549047)
    full/Gen range=(0.9273455520532593, 1.0262828817770455), (F/B)/(F/B)_Gen range=(0.9868248531242326, 1.1066887413270345)
    full/Gen-eta-dep range=(0.9722031341997304, 1.0070965863970451), (F/B)/(F/B)_Gen-eta-dep range=(0.9869442393792908, 1.0230705999842473)
  Gen (JER #eta-dep.)         full width integral=0.2, F/B range=(0.8632578003586374, 1.0330040154011362)
    full/Gen range=(0.9538598667619029, 1.0318212402598188), (F/B)/(F/B)_Gen range=(0.9940454415673462, 1.0817325229989747)
  Reco                        full width integral=0.2, F/B range=(0.9369809775308929, 1.06453217087547)
    full/Gen r